In [ ]:
# Load anndata object and create pseudobulk for MOFA analysis
# Necessary cell annotations in the anndata object are: 'cluster_id' (giving the cell-type cluster annotations), 'sample_id' (unique identifier of a sample)

# Prerequisites - Load Libraries

In [ ]:
from IPython.display import display, HTML

In [ ]:
from MS1_Functions import popup_function_pos

In [ ]:
from MS1_Functions import popup_function_info

In [ ]:
from MS1_Functions import popup_function_neg

In [ ]:
# Inform about execution start
popup_function_pos("01_Prepare_Pseudobulk.ipynb: Execution Started")

In [ ]:
import scanpy as sc

In [ ]:
import anndata as  ad

In [ ]:
import pandas as pd

In [ ]:
import random

In [ ]:
import numpy as np

In [ ]:
import random

In [ ]:
import os

In [ ]:
import decoupler as dc

In [ ]:
import time

In [ ]:
ad.__version__

In [ ]:
import numba

In [ ]:
print(numba.__version__)

In [ ]:
np.__version__

In [ ]:
import scanpy as sc
from scipy.sparse import issparse

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

# Preqrequisites Configurations & Parameters

In [ ]:
### Load the parameters that are set via the configuration files

In [ ]:
### Load configurations file
global_configs = pd.read_csv('configurations/Data_Configs.csv', sep = ',')

In [ ]:
global_configs

In [ ]:
data_path = global_configs['value'][global_configs['parameter'] == 'data_path']

In [ ]:
data_path

In [ ]:
result_path = global_configs['value'][global_configs['parameter'] == 'result_path']

In [ ]:
print(result_path.iloc[0])

In [ ]:
## Loading the file containing the name of the single-cell dataset

In [ ]:
sc_configs = pd.read_csv('configurations/01_Pre_Processing_SC_Data.csv', sep = ',')

In [ ]:
sc_configs

In [ ]:
sc_configs = sc_configs[sc_configs['data_name'] != '']

In [ ]:
sc_dataset_names = pd.unique(sc_configs['data_name'])

In [ ]:
sc_dataset_names

In [ ]:
### Generate the result data directory if it does not exist yet

if not os.path.exists(result_path[1] + '/01_results'):
    # Create the directory if it doesn't exist
    os.makedirs(result_path[1] + '/01_results')


# Load data

## Anndata object

In [ ]:
### Load single-cell datasets as anndata format; should contain the meta-columns: sample_id; cluster_id
### Raw Counts should be given in default assay

In [ ]:
sc_data_list = {}

In [ ]:
for i in sc_dataset_names:
    source_text = os.path.join(data_path[0], f'{i}'+ '.h5ad')
    
    if os.path.exists(source_text):
        print(source_text)
        print(os.path.getmtime(source_text))
        adata = ad.read_h5ad(source_text)
        #adata.raw.to_adata()

        # Store the loaded Seurat object in the dictionary
        sc_data_list[i] = adata
        popup_function_pos(" Data Loaded")
    else:
        print(f"The file '{  source_text}' does not exist.")
        popup_function_neg("Error: the single-cell data file could not be loaded - check the entered path and name of the dataset in the configuration files")

In [ ]:
adata

In [ ]:
adata.obs

# Data-Checks

In [ ]:
# Optional execute some checks to see whether raw counts will be used

In [ ]:
### Check that data contains the required columns

In [ ]:
if 'cluster_id' in adata.obs.columns and 'sample_id' in adata.obs.columns:
    print('Data correctly formatted')
else:
    print('cluster_id or sample_id meta data columns are missing in the anndata make sure to include them')
    popup_function_neg("Error: cluster_id or sample_id meta data columns are missing in the anndata make sure to include them")
    

## Check amount of cells per sample and cluster

In [ ]:
cell_counts_per_cluster = {key: pd.crosstab(value.obs['cluster_id'], value.obs['cluster_id']).sum(axis=1) for key, value in sc_data_list.items()}

In [ ]:
### Amount of cells per cluster
cell_counts_per_cluster

In [ ]:
### Save the result as a dataframe for usage in the next script
cells_per_sample_cluster = {key: pd.crosstab(value.obs['cluster_id'], value.obs['sample_id']).T for key, value in sc_data_list.items()}

In [ ]:
# Convert the dictionary of DataFrames into a single DataFrame and melt it to long format
cells_per_sample_cluster_long = pd.concat({key: value.melt(ignore_index=False) for key, value in cells_per_sample_cluster.items()})

# Reset index and rename columns
cells_per_sample_cluster_long.reset_index(inplace=True)
cells_per_sample_cluster_long.columns = ['dataset', 'Sample', 'Cluster_Cell_Type', 'amount_cells']

In [ ]:
cells_per_sample_cluster_long

In [ ]:
## Save the file
for i in pd.unique(cells_per_sample_cluster_long['dataset']):
    save_data = cells_per_sample_cluster_long[cells_per_sample_cluster_long['dataset'] == i]
    save_data.to_csv(result_path[1] + '01_results/01_' + i +  '_Cell_Sample_Cluster_Distribution.csv')

In [ ]:
## Categorize the amount of cells

In [ ]:
# Define bins and labels
bins = [-np.inf, 3, 10, 20, 50, np.inf]
labels = ['0-3', '3-10', '10-20', '20-50', '> 50']
# Map amount_cells to categories
cells_per_sample_cluster_long['amount_cells_cat'] = pd.cut(cells_per_sample_cluster_long['amount_cells'], bins=bins, labels=labels, right=False)

In [ ]:
cells_per_sample_cluster_long

In [ ]:
# Visualized the result

In [ ]:
my_colors = ['#8B0000', '#FFC1C1', '#FFF5EE', '#98FB98', '#548B54']
my_cmap = ListedColormap(my_colors)

bounds = [0, 3, 10, 20, 50, 10000]
my_norm = BoundaryNorm(bounds, ncolors=len(my_colors))

#grid_kws = {"height_ratios": (1), "hspace": .1}
# Adjust figsize for a less long and wider plot
fig, (ax, cbar_ax) = plt.subplots(nrows=2, figsize=(20, 10), gridspec_kw={"height_ratios": (15, 0.5), "hspace": .5})
sns.heatmap(data=cells_per_sample_cluster_long.pivot(index='Cluster_Cell_Type', columns='Sample', values='amount_cells'),
            #yticklabels=2, 
            ax=ax,
            cmap=my_cmap,
            norm=my_norm,
            cbar_ax=cbar_ax, xticklabels=True, yticklabels=True,
            cbar_kws={"orientation": "horizontal",  "pad": 0.05})


colorbar = ax.collections[0].colorbar
colorbar.set_ticks([(b0+b1)/2 for b0, b1 in zip(bounds[:-1], bounds[1:])])
colorbar.set_ticklabels(['0-3', '3-10', '10-20', '20-50', '>50'])

# Add title
plt.title("Amount of cells per cluster and sample")

# Save the figure as a PDF to a specific folder
plt.savefig('figures/01_figures/FIG01_Amount_of_Cells_per_Sample_and_Cell_Type'  + '.pdf', bbox_inches='tight')
plt.show()

popup_function_pos("FIG01_Amount_of_Cells_per_Sample_and_Cell_Type generated")

## Calculate gene expression percentages per cluster for thresholding

In [ ]:
### Calculate the percentage of cells having non-zero values for each gene per cluster
### will later be used to remove genes that are only expressed in a very low amount of cells

In [ ]:
## Generate empty data.frames to store the result in

In [ ]:
gene_expr_data = pd.DataFrame()

In [ ]:
for j in sc_data_list.keys():
    adata = sc_data_list[j]
    
    for i in pd.unique(adata.obs['cluster_id']):
        cell_type = i
        adata_subset = adata[adata.obs['cluster_id'] == cell_type]

        amount_cells = adata_subset.shape[0]
        ## Calcalte percentage of cells expressiong gene
        amount_cells_expressing_gene = (adata_subset.X > 0).sum(axis= 0)
        perc_cells_expressing_gene = (amount_cells_expressing_gene/ amount_cells) * 100

        data = {
        'perc_cells_expressing_gene': np.ravel(perc_cells_expressing_gene),
        'total_amount_cells_expressing_gene': np.ravel(amount_cells_expressing_gene),
        'gene': adata_subset.var_names,
        'cluster': cell_type,
        'dataset': j
        }

        df = pd.DataFrame(data)

        ### Append data
        gene_expr_data = pd.concat([gene_expr_data, df])

In [ ]:
gene_expr_data

In [ ]:
### Save results for later filtering
for i in pd.unique(gene_expr_data['dataset']):
    save_data = gene_expr_data[gene_expr_data['dataset'] == i]
    save_data.to_csv(result_path[1] + '/01_results/01_'  + i +  '_Gene_Expr_per_Cell_Type' +  '.csv')
    popup_function_pos('01_'  + i +  '_Gene_Expr_per_Cell_Type' +  '.csv' + " generated")

## Aggregate to pseudobulk

In [ ]:
## Aggregate to pseudobulk by cluster and sample_id

In [ ]:
adata.raw

In [ ]:
for j in sc_data_list.keys():
    adata = sc_data_list[j]
    popup_function_pos("Pseudobulk Aggregation started (this might take some time)")
    adata = dc.get_pseudobulk(
        adata,
        sample_col='sample_id',
        groups_col='cluster_id',
        #layer='counts',
        mode='mean',
        min_cells=0,  # Filter to remove samples by a minimum number of cells in a sample-group pair.
        min_counts=0)   
    
    ## Format as table 
    
    data = adata.to_df()
    data['feature'] = data.index
    
    ## Convert to long format
    data_long = pd.melt(data, id_vars = ['feature'])
    data_long['sample_id'] = data_long['feature'].str.extract(r'^([^_]*)')
    data_long['type'] = data_long['feature'].str.extract(r'_(.*)')
    
    if 'variable' in data_long.columns:
        data_long['variable'] = data_long['variable']
        
    if 'feature_name' in data_long.columns:
        data_long['variable'] = data_long['feature_name']    
    
    data_long['dataset'] = j
    popup_function_pos("Pseudobulk Aggregation finished")
    
    ## Select relevant columns and save
    data_long = data_long[['sample_id', 'variable', 'value', 'dataset', 'type']]
    data_long.to_csv(result_path[1] + '/01_results/01_' + j + 'Pseudobulk_Table' +'.csv')
    popup_function_pos('01_' + j + 'Pseudobulk_Table' +'.csv' + ' generated')

In [ ]:
j

In [ ]:
sc_dataset_names

In [ ]:
data_long

In [ ]:
len(pd.unique(data_long['sample_id']))

# Use information about cell types to adjust 02_Pre_Processing_Configs_SC File

In [ ]:
### Get configuration name

In [ ]:
configuration_name = global_configs['value'][global_configs['parameter'] == 'configuration_name'][2]

In [ ]:
configuration_name

In [ ]:
## Extract data names from data_long

In [ ]:
data_names = pd.unique(data_long['dataset'])

In [ ]:
data_names

In [ ]:
cell_counts_per_cluster.keys()

In [ ]:
## Use cell_counts_per_cluster to define cell-type exclustion

In [ ]:
records = []
for cluster_name, data in cell_counts_per_cluster.items():
    # Filter out clusters with cell counts below 50
    exclude = data[data < 50]
    # Get the cluster IDs from the index and join them into a single string
    cluster_ids = ','.join(exclude.index.tolist())
    
    # Create a record (a dictionary) for this iteration
    record = {
        'configuration_name': configuration_name,
        'data_name': cluster_name,
        'data_type': 'h5ad',
        'cell_type_exclusion': cluster_ids
    }
    records.append(record)

# Combine all records into one DataFrame
df = pd.DataFrame(records)
df.index.name = 'sc_config' 

'''
for i in cell_counts_per_cluster.keys():
    data = cell_counts_per_cluster[i]
    exclude = data[data < 50]
    cluster_id =  [ idx for idx in exclude.index]
    data = {
        
    'configuration_name': configuration_name,
    'data_name': i,
    'data_type': 'h5ad',
    'cell_type_exclusion':  ','.join(cluster_id) }
    df = pd.DataFrame(data,index = {'sc_config': 'sc_config'})
'''  

In [ ]:
df

In [ ]:
### Set default expression cutoffs

In [ ]:
#df['cell_expr_thres1'] = '50;10'
#df['cell_expr_thres2'] = '40;20'
df['cell_expr_thres1'] = '20;5'
df['cell_expr_thres2'] = '20;5'


In [ ]:
df

In [ ]:
### Write as new configuration file
df.to_csv('configurations/02_Pre_Processing_Configs_SC.csv', index = False)

In [ ]:
### Inform about successful execution
popup_function_pos("01_Prepare_Pseudobulk.ipynb: Execution Finished")

In [ ]:
time.sleep(20)
popup_function_info("01_Prepare_Pseudobulk.ipynb")